In [109]:
import meshio as mio
import h5py
import numpy as np
import pyvista as pv
from ipywidgets import interact, interactive, fixed, interact_manual
from matplotlib.widgets import Button, Slider
from scipy.spatial.transform import Rotation
import ipywidgets as widgets
import scipy as sp
import matplotlib.pyplot as mplt
import matplotlib
import matplotlib.animation as animation
import json as js
import meshio as mio
import subprocess as sup
import math
import tifffile
from scipy.fft import fft, ifft, fftfreq
from scipy.interpolate import RBFInterpolator
import igl
from tqdm import tqdm
pv.set_jupyter_backend('trame')

In [110]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    if F.shape[1] == 3:
        return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)
    elif F.shape[1] == 4:
        return pv.UnstructuredGrid({pv.CellType.TETRA: F}, V)

In [111]:
image = tifffile.imread("/Users/zoeli/Documents/UVic/masters/other/fish/Mask_3.tif")
#image_2 = tifffile.imread("/Users/zoeli/Documents/UVic/masters/other/fish/poor_quality_mask.tif")

print(image[180])
#print(image_2[180])

[[254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]
 ...
 [254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]]


In [112]:
# path = "/Users/teseo/Downloads/Embryogram test/new/0in-analysis-07_09T15_10-analysis.hdf5"
# path = "/Users/teseo/Downloads/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"
# path = "/Users/zoeli/Documents/UVic/masters/other/fish/still.hdf5"
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/neweststill.hdf5"

#o ct 16
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/not_control.hdf5"

#o ct 10
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/realone.hdf5"

# new 
path = "/Users/zoeli/Documents/UVic/masters/other/fish/20250523.hdf5"

# new 
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/poor_quality.hdf5"
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/20250702.hdf5"

#polyfem = "/Users/teseo/Documents/scuola/polyfem/polyfem.nosync/bin_rel.nosync/PolyFEM_bin"

In [113]:
hdf5_file = h5py.File(path, "r")
V, T = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)

top = hdf5_file["bc/top"][:].astype(np.int32)
bottom = hdf5_file["bc/bottom"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)

print(top, bottom, middle)

[   900    901    902 ... 170540 171364 171856] [  1800   1801   1802 ... 174015 177116 177158] [     0      1      2 ... 177097 177143 177144]


In [114]:
centers = hdf5_file["bc_func/centers"][:].astype(float)
eps = hdf5_file["bc_func/eps"][()]

In [115]:
nk = len(hdf5_file["bc_func"].keys())-2

disps = []

for i in range(nk):
    disps.append(hdf5_file[f"bc_func/disp{i+1}"][:].astype(float))

print(len(disps))

231


In [116]:
E = hdf5_file["problem/E"][()] .astype(float)
nu = hdf5_file["problem/nu"][()] .astype(float)
is_linear = hdf5_file["problem/is_linear"][()] .astype(bool)

# E

In [258]:
f = np.empty((0, 3), dtype=np.int32)
#for every face in a tet in T, check if all the vertices are in middle. If so, add the face to a list of faces to keep. Then remove all the unreferenced vertices and faces from the mesh.
for i in tqdm(range(T.shape[0])):
    tet = T[i, :]
    faces = [[tet[0], tet[1], tet[2]], [tet[0], tet[1], tet[3]], [tet[0], tet[2], tet[3]], [tet[1], tet[2], tet[3]]]
    for face in faces:
        if all(v in middle for v in face):
            f = np.vstack((f, face))

vv, ff, _, _ = igl.remove_unreferenced(V, f)

100%|██████████| 925404/925404 [00:11<00:00, 83692.19it/s]


In [259]:
new_ff = set()
keep = []

for face in ff:
    keep.append(not (frozenset({face[0], face[1], face[2]}) in new_ff))
    new_ff.add(frozenset({face[0], face[1], face[2]}))

ff = np.array(ff)[np.array(keep)]
keep = [True] * len(ff)


for i in tqdm(range(len(ff))):
    if not keep[i]:
        continue

    face = ff[i]
    frozen_face = frozenset({face[0], face[1], face[2]})

    for j in range(i, len(ff)):
        if not keep[j]:
            continue

        other = ff[j]
        frozen_other = frozenset({other[0], other[1], other[2]})
        intersection = frozen_face & frozen_other

        if len(intersection) == 2:
            shared_edge = [vv[k][:2] for k in list(intersection)]
            face_unique = np.array(vv[list(frozen_face - intersection)[0]][:2])
            other_unique = np.array(vv[list(frozen_other - intersection)[0]][:2])

            a = np.array([(shared_edge[0][0] + shared_edge[1][0]) / 2.0, (shared_edge[0][1] + shared_edge[1][1]) / 2.0])
            face_vec = face_unique - a
            other_vec = other_unique - a

            if np.dot(face_vec, other_vec) > 0:
                keep[i] = False


ff = np.array(ff)[np.array(keep)]

cells = [("triangle", ff)]
mesh = mio.Mesh(vv, cells)
path = "/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_0.obj"
mesh.write(path)

new_ff = set()

for face in ff:
    new_ff.add(frozenset({face[0], face[1], face[2]}))

#for i, disp in enumerate(disps):
#    current_points = centers + corrected_disps[i]
#    mask = [image[i][int(point[1] / 0.325)][int(point[0] / 0.325)] for point in current_points]
#    mesh = mio.Mesh(centers, cells, point_data={"corrected_disps": corrected_disps[i], "new_disps": new_disps[i], "mask": mask})
#    path = "/Users/zoeli/Documents/UVic/masters/other/fish/vtuFiles/disps" + str(i) + ".vtu"
#    mesh.write(path)
#print(ff)

100%|██████████| 11718/11718 [00:43<00:00, 270.95it/s] 


In [260]:
new_faces = set()

for i in tqdm(range(len(ff))):
    inds_0 = np.argwhere(ff == ff[i][0])[:,0]
    inds_1 = np.argwhere(ff == ff[i][1])[:,0]
    inds_2 = np.argwhere(ff == ff[i][2])[:,0]

    share_one_vertex = np.setxor1d(np.setxor1d(np.setxor1d(inds_0, inds_1), inds_2), np.array([i]))

    for ind in share_one_vertex:
        inds_0 = np.argwhere(ff == ff[ind][0])[:,0]
        inds_1 = np.argwhere(ff == ff[ind][1])[:,0]
        inds_2 = np.argwhere(ff == ff[ind][2])[:,0]

        ind_share_one_vertex = np.setxor1d(np.setxor1d(np.setxor1d(inds_0, inds_1), inds_2), np.array([ind]))

        for ind_ind in np.intersect1d(ind_share_one_vertex, share_one_vertex):
            if len(np.intersect1d(np.intersect1d(ff[ind_ind], ff[ind]), ff[i])) > 0:
                continue
            
            new_face = frozenset({np.intersect1d(ff[i], ff[ind])[0], np.intersect1d(ff[ind], ff[ind_ind])[0], np.intersect1d(ff[ind_ind], ff[i])[0]})
            
            if new_face not in new_ff:
                new_faces.add(new_face)

eek = np.array([ff[0]])

for face in new_faces:
    eek = np.append(eek, np.array([list(face)]), axis=0)

cells = [("triangle", eek)]
mesh = mio.Mesh(vv, cells)
path = "/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_1.obj"
mesh.write(path)

100%|██████████| 11573/11573 [00:34<00:00, 330.73it/s]


In [265]:
for i in tqdm(range(len(ff))):
    face = ff[i]
    if np.cross(vv[face][1] - vv[face][0], vv[face][2] - vv[face][0])[2] < 0:
        ff[i] = np.array([face[2], face[1], face[0]])

cells = [("triangle", ff)]
mesh = mio.Mesh(vv, cells)
path = "/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj"
mesh.write(path)

100%|██████████| 11694/11694 [00:00<00:00, 56009.31it/s]


In [261]:
for face in new_faces:
    ff = np.append(ff, np.array([list(face)]), axis=0)

cells = [("triangle", ff)]
mesh = mio.Mesh(vv, cells)
path = "/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_new.obj"
mesh.write(path)

In [ ]:
print(centers.shape)
first_frame = image[180]
plt = pv.Plotter()
mesh = to_pyvista_mesh(centers)
mesh['mask'] = [first_frame[int(center[1] / 0.325)][int(center[0] / 0.325)] for center in centers]
print(int(centers[0][0] / 0.325))
plt.add_mesh(mesh, point_size=7, render_points_as_spheres=True, scalars='mask', cmap='winter')
plt.show()

(900, 3)
538


Widget(value='<iframe src="http://localhost:59968/index.html?ui=P_0x17596496f00_15&reconnect=auto" class="pyvi…

[[254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]
 ...
 [254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]
 [254 254 254 ... 254 254 254]]


In [76]:
neighbourhoods = []

for i, disps_per_frame in enumerate(disps):
    neighbourhoods_per_frame = []

    for j, disp in enumerate(disps_per_frame):
        distances = np.linalg.norm(centers - centers[j], axis=1)
        indices = np.argsort(distances)
        neighbourhood = indices[1:19]

        neighbourhoods_per_frame.append(neighbourhood)
    
    neighbourhoods.append(neighbourhoods_per_frame)

In [77]:
corrected_disps = np.array(disps)
corrected = [[0.0 for disp in disps_per_frame] for disps_per_frame in disps]

count = 1
num_time = 0

while count > 0 and num_time < 10:
    count = 0

    for i, disps_per_frame in enumerate(corrected_disps):
        for j, disp in enumerate(disps_per_frame):
            neighbourhood_disps = disps_per_frame[neighbourhoods[i][j]]

            neighbourhood_mean = np.mean(neighbourhood_disps, axis=0)
            neighbourhood_lengths = np.linalg.norm(neighbourhood_disps, axis=1)
            neighbourhood_length_mean = np.mean(neighbourhood_lengths)
            cosine_similarity = np.dot(neighbourhood_mean, disp) / (np.linalg.norm(neighbourhood_mean) * np.linalg.norm(disp))
            length_difference = abs(neighbourhood_length_mean - np.linalg.norm(disp))
            difference = np.linalg.norm(neighbourhood_mean - disp)

            if length_difference > 4.0:
                corrected_disps[i][j] = neighbourhood_mean
                corrected[i][j] = 1.0
                count += 1
            elif difference > 4.0:
                corrected_disps[i][j] = neighbourhood_mean
                corrected[i][j] = 1.0
                count += 1
            elif cosine_similarity < 0.9:
                corrected_disps[i][j] = neighbourhood_mean
                corrected[i][j] = 1.0
                count += 1
    
    num_time += 1
    print(num_time)

1
2
3
4
5
6
7
8
9
10


In [86]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])
corrected_lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
ll['corrected'] = corrected[0]
pl.add_mesh(ll, name='line', scalars='corrected', cmap='bwr')

corrected_vertices = np.vstack([centers, centers + corrected_disps[0]])
corrected_ll = pv.PolyData(corrected_vertices, lines=corrected_lines)
corrected_ll['corrected'] = corrected[0]
pl.add_mesh(corrected_ll, name='corrected_line', scalars='corrected', cmap='winter')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    ll['corrected'] = corrected[x]
    pl.add_mesh(ll, name='line', scalars='corrected', cmap='bwr')

    corrected_vertices = np.vstack([centers, centers + corrected_disps[x]])
    corrected_ll = pv.PolyData(corrected_vertices, lines=corrected_lines)
    corrected_ll['corrected'] = corrected[x]
    pl.add_mesh(corrected_ll, name='corrected_line', scalars='corrected', cmap='winter')
    
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))



Widget(value='<iframe src="http://localhost:59968/index.html?ui=P_0x175f82553a0_8&reconnect=auto" class="pyvis…

interactive(children=(IntSlider(value=120, description='x', max=240), Output()), _dom_classes=('widget-interac…

<function __main__.callback(x)>

In [89]:
pl = pv.Plotter()
mesh = to_pyvista_mesh(centers)
mesh['mask'] = [image[0][int(center[1] / 0.325)][int(center[0] / 0.325)] for center in centers]
pl.add_mesh(mesh, point_size=7, render_points_as_spheres=True, scalars='mask', cmap='winter', name="uhhh")

def callback(x):
    current_points = centers + corrected_disps[x]
    mesh = to_pyvista_mesh(current_points)
    mesh['mask'] = [image[x][int(center[1] / 0.325)][int(center[0] / 0.325)] for center in current_points]
    pl.add_mesh(mesh, point_size=7, render_points_as_spheres=True, scalars='mask', cmap='winter', name="uhhh")
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))

Widget(value='<iframe src="http://localhost:59968/index.html?ui=P_0x175f8362210_11&reconnect=auto" class="pyvi…

interactive(children=(IntSlider(value=120, description='x', max=240), Output()), _dom_classes=('widget-interac…

<function __main__.callback(x)>

In [91]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=7)
knn.fit(centers[:,:2])
neighbors = knn.kneighbors(centers[:,:2], return_distance=False)

In [92]:
from tqdm import tqdm

new_disps = [p.copy() for p in corrected_disps]
n_frames = len(disps)
n_pts = len(centers)

for it in tqdm(range(200)):
    tmp = [p.copy() for p in new_disps]

    for fr in range(1, n_frames - 1):
        for j in range(n_pts):
            current_point = centers[j] + corrected_disps[fr][j]
            
            try:
                if image[fr][int(current_point[1] / 0.325)][int(current_point[0] / 0.325)] < 100:
                    continue
            except:
                continue

            neighbors_idxs = neighbors[j]
            avg_dispi = np.mean(new_disps[fr][neighbors_idxs], axis=0)
            avg_dispip = np.mean(new_disps[fr+1][neighbors_idxs], axis=0)
            avg_dispim = np.mean(new_disps[fr-1][neighbors_idxs], axis=0)
            # print(avg_dispi)
            tmp[fr][j] = (avg_dispi + avg_dispim + avg_dispip) / 3

    #print(np.array(new_disps[0]) - np.array(tmp[0]))
    #uhh = [np.average(np.linalg.norm(np.array(new_disps[i]) - np.array(tmp[i]), axis = 1)) for i in range(len(new_disps))]
    #print(np.average(uhh))
    new_disps = tmp

100%|██████████| 200/200 [03:48<00:00,  1.14s/it]


In [93]:
final_disps = corrected_disps - new_disps

In [94]:
pl = pv.Plotter()
mesh = to_pyvista_mesh(centers)
mesh['mask'] = [disp[0] for disp in final_disps[0]]
pl.add_mesh(mesh, point_size=7, render_points_as_spheres=True, scalars='mask', cmap='winter', name="uhhh")

def callback(x):
    current_points = centers + new_disps[x]
    mesh = to_pyvista_mesh(centers)
    mesh['mask'] = [disp[2] for disp in final_disps[x]]
    pl.add_mesh(mesh, point_size=7, render_points_as_spheres=True, scalars='mask', cmap='winter', name="uhhh")
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))

Widget(value='<iframe src="http://localhost:59968/index.html?ui=P_0x1760ff40fb0_13&reconnect=auto" class="pyvi…

interactive(children=(IntSlider(value=120, description='x', max=240), Output()), _dom_classes=('widget-interac…

<function __main__.callback(x)>

In [ ]:
cells = [("vertex", np.array([[i,] for i in range(len(centers))]))]

for i, disp in enumerate(disps):
    current_points = centers + corrected_disps[i]
    mask = [image[i][int(point[1] / 0.325)][int(point[0] / 0.325)] for point in current_points]
    mesh = mio.Mesh(centers, cells, point_data={"corrected_disps": corrected_disps[i], "new_disps": new_disps[i], "mask": mask})
    path = "/Users/zoeli/Documents/UVic/masters/other/fish/vtuFiles/disps" + str(i) + ".vtu"
    mesh.write(path)

In [95]:
cells = [("vertex", np.array([[i,] for i in range(len(centers))]))]

for i, disp in enumerate(disps):
    mesh = mio.Mesh(centers, cells, point_data={"disps": final_disps[i]})
    path = "/Users/zoeli/Documents/UVic/masters/other/fish/poor_quality_disps/disps" + str(i) + ".vtu"
    mesh.write(path)

In [ ]:
rbf_disps = []

for i, disp in enumerate(disps):
    rbf = RBFInterpolator(centers, corrected_disps[i] - new_disps[i])
    rbf_disp = rbf(centers)
    rbf_disps.append(rbf_disp)

In [ ]:
sorting_indexes = np.lexsort((centers[:,0], centers[:,1]))

cut = sorting_indexes[0:45]
i = 90
while (i < len(sorting_indexes)):
    if i % 180 != 45:
        i += 45
        continue
    
    cut = np.concatenate((cut, sorting_indexes[i - 45:i]))
    i += 45

sorting_indexes = np.lexsort((centers[:,1], centers[:,0]))

cut2 = sorting_indexes[0:20]
i = 40
while (i < len(sorting_indexes)):
    if i % 80 != 20:
        i += 20
        continue
    
    cut2 = np.concatenate((cut2, sorting_indexes[i - 20:i]))
    i += 20

finalcut = np.intersect1d(cut, cut2)

pl = pv.Plotter()
mesh = to_pyvista_mesh(centers[finalcut])
pl.add_mesh(mesh, point_size=7, render_points_as_spheres=True)
pl.show()

rbf_disps_fourth = []

for i, disp in enumerate(disps):
    rbf = RBFInterpolator(centers[finalcut], (corrected_disps[i] - new_disps[i])[finalcut])
    rbf_disp = rbf(centers)
    rbf_disps_fourth.append(rbf_disp)

In [ ]:
pl = pv.Plotter()
mesh = to_pyvista_mesh(centers)
mesh['mask'] = [disp[0] for disp in rbf_disps[0]]
pl.add_mesh(mesh, point_size=7, render_points_as_spheres=True, scalars='mask', cmap='winter', name="uhhh")

def callback(x):
    mesh = to_pyvista_mesh(centers)
    mesh['mask'] = [disp[2] for disp in rbf_disps[x]]
    pl.add_mesh(mesh, point_size=7, render_points_as_spheres=True, scalars='mask', cmap='winter', name="uhhh")
    pl.update()

pl.show()
interact(callback, x=(0, len(rbf_disps)-1, 1))

In [ ]:
cells = [("vertex", np.array([[i,] for i in range(len(centers))]))]

for i, disp in enumerate(disps):
    current_points = centers + corrected_disps[i]
    mask = [image[i][int(point[1] / 0.325)][int(point[0] / 0.325)] for point in current_points]
    mesh = mio.Mesh(centers, cells, point_data={
        "corrected_disps": corrected_disps[i], 
        "new_disps": new_disps[i], 
        "mask": mask, 
        "rbf_disps_half": rbf_disps[i], 
        "rbf_disps_third": rbf_disps_third[i], 
        "rbf_disps_fourth": rbf_disps_fourth[i]
    })
    path = "/Users/zoeli/Documents/UVic/masters/other/fish/vtuFiles/disps" + str(i) + ".vtu"
    mesh.write(path)

In [96]:
from scipy.interpolate import RBFInterpolator


In [97]:
default_json = { 
"geometry": {
        "mesh": "___",
        "volume_selection": 1
    },
"boundary_conditions": {
    "dirichlet_boundary": "___"
},
"materials": {
        "E": E,
        "id": 1,
        "nu": nu,
        "type": "LinearElasticity" if is_linear else "NeoHookean"
},
"output": {
    "json": "___",
    "directory": "___",
    "paraview": {
        "file_name": "___",
        "surface": True,
        "options": {
            "material": True,
            "forces": True
        },
        "vismesh_rel_area": 10000000
    }
}
}

In [98]:
def generate_json(out, V, T, middle, bottom, top, centers, disps, index, eps):
    rbf = RBFInterpolator(centers, disps[index])
    disp = rbf(V[middle,:])

    with open(f"{out}disp_{index}.txt", "w") as f:
        for i in range(middle.shape[0]):
            f.write(f"{middle[i]} {disp[i, 0]} {disp[i, 1]} {disp[i, 2]}\n")
        for i in range(bottom.shape[0]):
            f.write(f"{bottom[i]} 0 0 0\n")
        for i in range(top.shape[0]):
            f.write(f"{top[i]} 0 0 0\n")

    mesh = mio.Mesh(points=V, cells={"tetra": T})
    mesh.write(f"{out}mesh.msh", file_format="gmsh")

    json = default_json.copy()
    json["geometry"]["mesh"] = f"mesh.msh"
    json["boundary_conditions"]["dirichlet_boundary"] = f"disp_{index}.txt"
    json["output"]["directory"] = out
    json["output"]["json"] = f"sim{index}.json"
    json["output"]["paraview"]["file_name"] = f"sim{index}.vtu"

    with open(f"{out}run_{index}.json", "w") as f:
        js.dump(json, f, indent=4)



#generate_json("outnz/", V,T, middle, bottom, top, centers, disps, 239, eps)

In [99]:
for i in range(len(disps)):
    generate_json("outnz/", V,T, middle, bottom, top, centers, final_disps, i, eps)

In [ ]:
index=200
sup.run([polyfem, "-j", f"outnz/run_{index}.json"], check=True)

In [ ]:
rbf_disps = []

for i, disp in enumerate(disps):
    corrected_tf = [val < 0.5 for val in corrected[i]]
    rbf = RBFInterpolator(centers[corrected_tf], new_corrected_disps[i][corrected_tf])
    rbf_disp = rbf(centers)
    rbf_disps.append(rbf_disp)

In [ ]:
pl = pv.Plotter()

pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = centers.shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

#corrected_tf = [val < 0.5 for val in corrected[0]]
#rbf = RBFInterpolator(centers[corrected_tf], new_disps[0][corrected_tf])
#rbf_disp = rbf(centers)

vertices = np.vstack([centers, centers + rbf_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='green')

pl.camera_position = 'xz'
pl.camera.elevation = 25
def callback(x):
    #corrected_tf = [val < 0.5 for val in corrected[x]]
    #rbf = RBFInterpolator(centers[corrected_tf], new_disps[x][corrected_tf])
    #rbf_disp = rbf(centers)

    vertices = np.vstack([centers, centers + rbf_disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='green')
    
    pl.update()
    #pl.camera_position = 'xy'

pl.show()
interact(callback, x=(0, len(disps)-1, 1))

In [ ]:
cells = [("vertex", np.array([[i,] for i in range(len(centers))]))]

for i, disp in enumerate(disps):
    mesh = mio.Mesh(centers, cells, point_data={
        "original_disps": disps[i], 
        "outlier_identification": corrected[i], 
        "corrected_pca_scale_translation": new_corrected_disps[i], 
        "corrected_scale_translation": new_corrected_disps_only_scale_and_translation[i], 
        "corrected_pca": new_corrected_disps_only_pca[i], 
       # "corrected_pca_scale_translation_with_smoothing": new_corrected_smoothed_disps[i], 
        "rbf": rbf_disps[i]
    })
    path = "/Users/zoeli/Documents/UVic/masters/other/fish/vtuFiles/disps" + str(i) + ".vtu"
    mesh.write(path)

In [ ]:
pl = pv.Plotter(notebook=False, off_screen=True)
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = centers.shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + rbf_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='green')

actor = pl.add_text(str(0), position='upper_right')
pl.camera_position = 'xz'
pl.camera.elevation = 25

# Open a gif
pl.open_gif("updated.gif")

# Update Z and write a frame for each updated position
nframe = 15
for index in range(len(rbf_disps)):
    vertices = np.vstack([centers, centers + rbf_disps[index]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='green')

    actor.set_text('upper_right', str(index))

    # Write a frame. This triggers a render.
    pl.write_frame()

# Closes and finalizes movie
pl.close()

In [ ]:
pl = pv.Plotter(notebook=False, off_screen=True)
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

actor = pl.add_text(str(0), position='upper_right')
pl.camera_position = 'xz'
pl.camera.elevation = 45

# Open a gif
pl.open_gif("test.gif")

# Update Z and write a frame for each updated position
nframe = 15
for index in range(len(rbf_disps)):
    pl.add_mesh(to_pyvista_mesh(centers + rbf_disps[index]), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

    actor.set_text('upper_right', str(index))

    # Write a frame. This triggers a render.
    pl.write_frame()

# Closes and finalizes movie
pl.close()

In [ ]:
pvsm = "C:/Users/zoeli/Documents/UVic/masters/other/fish/state.pvsm"
mesh = "C:/Users/zoeli/Documents/UVic/masters/other/fish/wildtype.obj"

for i in range(1, 4):
    sim = "C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim" + str(i) + ".vtu"
    LoadState(pvsm, filenames=[{"name": "sim110.vtu", "FileName": sim}, {"name": "wildtype.obj", "FileName": mesh}])
    my_source = FindSource("ResampleWithDataset1")
    SaveData("C:/Users/zoeli/Documents/UVic/masters/other/fish/outputest/sim" + str(i) + ".vtu", proxy=my_source)
    ResetSession()

C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim1.vtu
C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim2.vtu


In [285]:
tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim100.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim100.vtu")

In [286]:
tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim101.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim101.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim102.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim102.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim103.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim103.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim104.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim104.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim105.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim105.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim106.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim106.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim107.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim107.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim108.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim108.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim109.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim109.vtu")

tet_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/final/sim110.vtu")
boundary_mesh = mio.read("C:/Users/zoeli/Documents/UVic/masters/other/fish/high_quality_manifold.obj")
barycenters = igl.barycenter(tet_mesh.points, tet_mesh.cells_dict['tetra'])
winding_numbers = igl.fast_winding_number(boundary_mesh.points, boundary_mesh.cells_dict['triangle'], barycenters)
tet_mesh.cell_data["winding_numbers"] = [winding_numbers]
tet_mesh.write("C:/Users/zoeli/Documents/UVic/masters/other/fish/final2/sim110.vtu")